## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [2]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [5]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Anthropic AI")


In [6]:
# Here is the final output

print(result.final_output)

Anthropic AI walks into a bar and says, “I’d like something safe, helpful, and honest.”

The bartender replies, “Great. What’s your limit?”

Anthropic AI says, “I’m not allowed to disclose that.”


In [7]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Anthropic AI', 'role': 'user'},
 {'id': 'msg_04ccb1adfd6f4d8b006aa22534d11487d09cd347a06127326a',
  'content': [{'annotations': [],
    'text': 'Anthropic AI walks into a bar and says, “I’d like something safe, helpful, and honest.”\n\nThe bartender replies, “Great. What’s your limit?”\n\nAnthropic AI says, “I’m not allowed to disclose that.”',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Adding Observability with a trace

In [8]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Autonomous AI agents are like interns who never sleep, never ask for coffee, and still somehow send you a 47-step plan titled “Quick Improvement.”


## Now go and look at the trace

https://platform.openai.com/traces

In [9]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure — here are 5 AI Agent jokes:

1. Why did the AI agent get promoted?  
   Because it always followed through — unlike its human manager.

2. I asked my AI agent to be more proactive.  
   Now it sends me reminders about reminders.

3. Why did the AI agent break up with the chatbot?  
   It needed more space to process its feelings.

4. My AI agent said it could handle multitasking.  
   So I gave it five jobs. Now it’s just confidently confused in parallel.

5. What’s an AI agent’s favorite type of music?  
   Loop and bass.

If you want, I can also give you:
- darker AI jokes
- nerdier / more technical AI jokes
- workplace AI agent jokes

## Part 2: Adding a tool

In [12]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

keys = [("pushover_user", pushover_user, "u"), ("pushover_token", pushover_token, "a")]

for key_name, key_value, key_prefix in keys:
    if not key_value:
        print(f"{key_name} not found")
    elif not key_value.startswith(key_prefix):
        print(f"{key_name} found but doesn't start with {key_prefix}")
    else:
        print(f"{key_name} found and looks good")
        

pushover_user found and looks good
pushover_token found and looks good


In [13]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [14]:
push("HEY!!")

Push: HEY!!


In [15]:
push

<function __main__.push(message)>

In [16]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [17]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000002070B76EFC0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [19]:
push_tool.description

'Send the given message to the user as a push notification'

In [20]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [21]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Done.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [22]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [23]:
response = await Runner.run(agent, "Hi there. My name is Tolu.")
print(response.final_output)

Hi Tolu — nice to meet you! How can I help today?


In [24]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name yet. If you’d like, tell me what to call you.


## Memory approach 1 - just manually pass in the list of dicts

In [25]:
response = await Runner.run(agent, "Hi there. My name is Tolu.")
print(response.final_output)

Hi Tolu! Nice to meet you. How can I help today?


In [26]:
response.to_input_list()

[{'content': 'Hi there. My name is Tolu.', 'role': 'user'},
 {'id': 'msg_0c1547dd0c18fe1c006aa22fb74ea487d0a6c69f8f10ed38c6',
  'content': [{'annotations': [],
    'text': 'Hi Tolu! Nice to meet you. How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [27]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Tolu.', 'role': 'user'},
 {'id': 'msg_0c1547dd0c18fe1c006aa22fb74ea487d0a6c69f8f10ed38c6',
  'content': [{'annotations': [],
    'text': 'Hi Tolu! Nice to meet you. How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [28]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Tolu.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [29]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [30]:
response = await Runner.run(agent, "Hi there. My name is Tolu.", session=session)
print(response.final_output)

Hi Tolu, nice to meet you! How can I help today?


In [ ]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>